# Athena query module

Notebook này là giao diện điều khiển Athena sau khi dữ liệu đã được Glue ghi thành Parquet và partition theo `year/month/day`. Logic triển khai nằm trong `setup.py` để có thể tái sử dụng từ CLI và CI/CD.

Luồng dữ liệu:

```text
S3 curated Parquet → Glue Data Catalog → Athena workgroup → SQL query results trên S3
```

Notebook không tạo lại schema/table. Nó kiểm tra Glue table đã tồn tại rồi mới setup Athena workgroup.

## 1. Chuẩn bị môi trường

Tạo virtual environment và cài dependency trước khi chọn kernel:

```bash
cd athena
cp .env.example .env
python -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

Trên PowerShell, activate bằng `.\.venv\Scripts\Activate.ps1`. Sau đó điền đúng bucket và tên table thực tế từ Glue Data Catalog vào `.env`.

In [8]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

# Hỗ trợ mở notebook từ thư mục athena/ hoặc từ workspace root.
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "setup.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "athena"
if not (NOTEBOOK_DIR / "setup.py").exists():
    raise FileNotFoundError("Không tìm thấy athena/setup.py")

module_path = str(NOTEBOOK_DIR.resolve())
if module_path not in sys.path:
    sys.path.insert(0, module_path)

import setup as athena_setup

# Reload giúp notebook nhận code mới mà không cần restart kernel.
athena_setup = importlib.reload(athena_setup)
print(f"Loaded Athena module: {Path(athena_setup.__file__).resolve()}")

Loaded Athena module: D:\aws\Archiver-Data\athena\setup.py


## 2. Kiểm tra cấu hình

Cell này chỉ đọc và validate `.env`, chưa gọi AWS và chưa tạo tài nguyên. Không in credential vì module Athena không lưu credential trong `.env`.

In [9]:
cfg = athena_setup.load_config()
print(f"Region       : {cfg.region}")
print(f"Workgroup    : {cfg.workgroup}")
print(f"Results      : {cfg.output_location}")
print(f"Database     : {cfg.database}")
print(f"RDS table    : {cfg.rds_table}")
print(f"Dynamo table : {cfg.ddb_table}")
print(f"Scan cutoff  : {cfg.scan_cutoff_mb:,} MB/query")

Region       : ap-southeast-1
Workgroup    : archiver-data
Results      : s3://my-data-lake-archival-demo/athena-results/archiver-data/
Database     : archive
RDS table    : orders_rds_orders
Dynamo table : orders
Scan cutoff  : 1,024 MB/query


## 3. Tìm đúng tên Glue table

Cell dưới in AWS Account ID, Region, danh sách Glue database, rồi mới liệt kê table và S3 location. Nếu không thấy database `archive`, hãy đối chiếu account/Region với notebook partition; không nên tạo database rỗng bằng tay vì Athena vẫn chưa có schema để query.

Glue Crawler có thể ghép `TablePrefix` với tên suy ra từ S3 path, nên tên table thật có thể không phải `orders_rds`. Sau khi thấy database, sửa `RDS_GLUE_TABLE` và `DDB_GLUE_TABLE` trong `athena/.env`.

Table RDS phải trỏ vào `curated/rds/orders`; table DynamoDB phải trỏ vào `curated/dynamodb/orders`.

In [10]:
catalog_tables: list[dict[str, str]] = athena_setup.discover_tables()

AWS account  : 637423316258
AWS Region   : ap-southeast-1
Glue databases: (none)

Configured database 'archive' does not exist in this account/Region.
Check AWS_PROFILE/AWS_REGION or run the partition Glue crawler first.


## 4. Setup Athena

`setup()` thực hiện theo thứ tự:

1. Kiểm tra S3 bucket tồn tại và cùng Region.
2. Kiểm tra RDS và DynamoDB Glue tables đã tồn tại.
3. Tạo hoặc update Athena workgroup.
4. Enforce result location, SSE-S3 và giới hạn byte scan.
5. Tạo lại các saved query mẫu do module quản lý.

Lệnh được comment để **Run All không tự tạo resource**. Bỏ dấu `#` khi đã kiểm tra cấu hình.

In [3]:
athena_setup.setup()

RuntimeError: Glue table archive.orders_rds does not exist. Run the partition Glue job/crawler first and copy the exact table name from Glue Data Catalog into athena/.env.

## 5. Xem trạng thái

`status()` chỉ đọc workgroup, scan cutoff, Glue tables và saved queries. Có thể chạy nhiều lần mà không thay đổi tài nguyên.

In [ ]:
# athena_setup.status()

## 6. Smoke test RDS table

Hàm chạy `SELECT * ... LIMIT 10`, chờ query hoàn tất, in rows và lượng MB đã scan. `LIMIT` chỉ giới hạn số row trả về, **không đảm bảo giới hạn dữ liệu scan**; workgroup scan cutoff vẫn là lớp bảo vệ chính.

In [ ]:
# rds_rows: list[list[str]] = athena_setup.test_query("rds")

## 7. Smoke test DynamoDB table

Chỉ chạy sau khi DynamoDB export và Glue job đầu tiên đã hoàn tất.

In [ ]:
# ddb_rows: list[list[str]] = athena_setup.test_query("ddb")

## 8. Chạy SQL tùy chỉnh an toàn

Các saved query trong Athena Console đã có ví dụ filter đủ `year/month/day`. Luôn dùng partition filter để Athena chỉ đọc folder cần thiết:

```sql
SELECT status, COUNT(*) AS total_orders, SUM(amount) AS total_amount
FROM archive.orders
WHERE year = '2026' AND month = '08' AND day = '01'
GROUP BY status;
```

Nếu bỏ partition filter, Athena có thể scan toàn bộ lịch sử và tăng chi phí.

## 9. Destroy Athena workgroup

`destroy()` xóa workgroup và saved queries nhưng giữ nguyên Glue database/tables, dữ liệu raw/curated và các query-result object trên S3. Query results nên được dọn bằng S3 lifecycle riêng.

Lệnh được comment để tránh xóa nhầm khi Run All.

In [ ]:
# athena_setup.destroy()